# Vocalyze AI — Audio Intelligence Assistant

**Group 25 · AI Course Capstone Project**

Upload a meeting recording; get back a speaker-attributed transcript and a brief
in which **every point carries the line it came from**.

This notebook is the project's executable record. It runs the four components in
the order the system runs them, and each section is self-contained: run the
setup cell at the top, then any section on its own.

| # | Section | What it establishes |
|---|---|---|
| 1 | Data Acquisition & Benchmarking | AMI and ICSI loaded, speaker-disjoint splits, 16 kHz mono compliance |
| 2 | Speech Recognition (Whisper) | WER and latency on AMI |
| 3 | Speaker Diarization (pyannote) | DER on ICSI |
| 4 | Integration & Privacy | The service that merges the three into one verifiable output |
| 5 | Run it on your own recording | A public link, from this notebook, on a free GPU |

**Before running:** `Runtime → Change runtime type → T4 GPU`.

## Team and role assignments

| Member | Role | Sections |
|---|---|---|
| Turki Aljuhani | Team Leader, Diarization & Evaluation | 3 |
| Reema Alsamrani | Data & Benchmarking | 1 |
| Taghreed Alotaibi | ASR & Performance Engineer | 2 |
| Aljawharah Alnajim | Grounded LLM & Quality Assurance | 4.5 |
| Naif Aldosari | Application Integration & Privacy | 4 |
| Saad Alsharfaa | Planning, Documentation & Demo Support | 5 |

> Section headings below name the **role**, not the person, because the
> notebook's original section labels and the final report's role table did not
> agree. The table above is the one from the report; it is the authority.

---

## 0 · Environment

One install cell for the whole notebook. Run it once, then jump to any section.

In [ ]:
# Everything the notebook needs, installed once.
!pip install -q datasets huggingface_hub pandas
!pip install -q openai-whisper jiwer librosa
!pip install -q pyannote.audio soundfile
!apt-get -qq install -y ffmpeg > /dev/null

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime -> Change runtime type -> T4 GPU, or expect this to be slow.")

---

# 1 · Data Acquisition & Benchmarking

Two multi-speaker meeting corpora, acquired programmatically:

- **AMI** — the primary benchmark. Word-level alignments and ground-truth
  speaker timestamps, so it can score both ASR and diarization.
- **ICSI** — the secondary benchmark. Multi-party discussion with heavier
  natural overlap, used for diarization.

The evaluation question this section has to answer honestly is **leakage**: a
model that has heard a speaker in training will do better on that speaker at
test time, and the score stops meaning anything. So the split is
speaker-disjoint, not random over utterances.

### 1.1 · The AMI metadata

`remove_columns(["audio"])` is what makes this cheap: the speaker inventory
needs the labels, not ~100 hours of waveform.

In [ ]:
from datasets import get_dataset_config_names, load_dataset

print("AMI configs:", get_dataset_config_names("edinburghcstr/ami"))

# IHM = individual headset microphones: one clean channel per speaker.
metadata = load_dataset(
    "edinburghcstr/ami",
    "ihm",
    split="train",
    streaming=False,
).remove_columns(["audio"])

unique_speakers = {sample["speaker_id"] for sample in metadata}
print("Unique speakers:", len(unique_speakers))

### 1.2 · Speaker-disjoint 70 / 15 / 15 split

Every speaker belongs to exactly one split. The split sizes are *computed* from
the speaker count and then used — the original notebook computed them and then
sliced with hard-coded indices (`[:109]`, `[109:132]`), which silently stops
being a 70/15/15 split the moment the corpus changes.

In [ ]:
import random

speaker_list = sorted(unique_speakers)   # sorted first, so the shuffle is reproducible
random.seed(42)
random.shuffle(speaker_list)

total = len(speaker_list)
n_train = round(total * 0.70)
n_validation = round(total * 0.15)

train_speaker_ids = speaker_list[:n_train]
validation_speaker_ids = speaker_list[n_train : n_train + n_validation]
test_speaker_ids = speaker_list[n_train + n_validation :]

print(f"Total speakers:      {total}")
print(f"Train      (70%):    {len(train_speaker_ids)}")
print(f"Validation (15%):    {len(validation_speaker_ids)}")
print(f"Test       (15%):    {len(test_speaker_ids)}")

In [ ]:
# The split is only worth anything if it is actually disjoint. Assert it.
train, validation, test = set(train_speaker_ids), set(validation_speaker_ids), set(test_speaker_ids)

assert not (train & validation), "speaker leak: train / validation"
assert not (train & test), "speaker leak: train / test"
assert not (validation & test), "speaker leak: validation / test"
assert len(train | validation | test) == total, "a speaker was lost in the split"

print("Speaker-disjoint split verified: no speaker appears in more than one split.")

### 1.3 · 16 kHz mono standardisation

Whisper and pyannote both expect 16 kHz mono. Casting the column is lazy — the
resample happens when a sample is actually pulled, not now.

In [ ]:
from datasets import Audio

audio_dataset = load_dataset("edinburghcstr/ami", "ihm", split="train", streaming=True)
audio_dataset = audio_dataset.cast_column("audio", Audio(sampling_rate=16000))

sample = next(iter(audio_dataset))
print("Sampling rate:", sample["audio"]["sampling_rate"])
print("Array shape:  ", sample["audio"]["array"].shape)

### 1.4 · ICSI, and the sample validator

`audio_info` exists because the `audio` column has two shapes: a dict
(`{"array", "sampling_rate"}`) on a downloaded dataset, and a decoder object
with `.metadata` on a streaming one. A validator that reads only one shape
rejects every sample, and the DER loop in section 3 then reports zero meetings
without ever saying why.

**Section 3 depends on `is_valid_sample` and `icsi_dataset` from this cell.**

In [ ]:
icsi_dataset = load_dataset("argmaxinc/icsi-meetings", split="test", streaming=True)
print("ICSI fields:", list(next(iter(icsi_dataset)).keys()))


def audio_info(sample):
    """Return (duration_seconds, sample_rate, num_channels) for either shape."""
    audio = sample["audio"]

    if isinstance(audio, dict):
        array = audio["array"]
        rate = audio["sampling_rate"]
        channels = 1 if getattr(array, "ndim", 1) == 1 else array.shape[0]
        return len(array) / rate, rate, channels

    meta = audio.metadata
    return meta.duration_seconds, meta.sample_rate, meta.num_channels


def is_valid_sample(sample):
    """A sample is usable for DER only if audio and reference labels line up."""
    try:
        duration, _rate, _channels = audio_info(sample)
        if duration <= 0:
            return False
    except Exception:
        return False

    speakers = sample.get("speakers") or []
    starts = sample.get("timestamps_start") or []
    ends = sample.get("timestamps_end") or []

    if not speakers or not starts or not ends:
        return False
    return len(speakers) == len(starts) == len(ends)

In [ ]:
# 16 kHz / mono compliance across the corpus.
#
# Bounded on purpose: `icsi_dataset` is a streaming iterator over meeting-length
# audio, and the original unbounded loop downloads the entire corpus to print
# two numbers.
CHECK_LIMIT = 20

checked = non_compliant = invalid = 0
for sample in icsi_dataset:
    if checked >= CHECK_LIMIT:
        break
    checked += 1
    if not is_valid_sample(sample):
        invalid += 1
        continue
    _duration, rate, channels = audio_info(sample)
    if rate != 16000 or channels != 1:
        non_compliant += 1

print(f"Checked:              {checked}")
print(f"Not 16 kHz mono:      {non_compliant}")
print(f"Unusable for DER:     {invalid}")

---

# 2 · Speech Recognition — Whisper

**Target: WER ≤ 10%.**

Two decisions that determine whether the number means anything:

**Normalisation before comparison.** Whisper writes "Let's start." and the AMI
reference is "LET'S START" — punctuation and case make an identical
transcription look like errors. Both sides are lowercased and stripped of
punctuation first.

**Degenerate samples excluded.** AMI IHM contains one- and two-word utterances
("Mm-hmm", "Yeah"). A single substitution there is a WER of 1.0, and a handful
of them drags the corpus average far above what the model actually does. They
are excluded, and the count of exclusions is printed rather than hidden.

In [ ]:
import re
import time

import librosa
import whisper
from jiwer import wer

# One model, loaded once, used by every cell in this section. The original
# notebook loaded "small" here and "base" again further down, so the WER figure
# and the latency figure came from different models.
WHISPER_MODEL = "small"

whisper_model = whisper.load_model(WHISPER_MODEL, device=DEVICE)
print(f"Loaded Whisper '{WHISPER_MODEL}' on {DEVICE}")


def normalize_text(text):
    """Lowercase, strip punctuation, collapse whitespace."""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def is_degenerate_sample(reference_text, min_words=3):
    """Backchannels are too short for WER to be meaningful on them."""
    return len(reference_text.split()) < min_words


def whisper_asr(audio_array, sample_rate):
    """Transcribe one clip. Returns (text, seconds_taken)."""
    if sample_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=16000)
    started = time.time()
    result = whisper_model.transcribe(audio_array, language="en")
    return result["text"], time.time() - started

### 2.1 · WER and latency on AMI

One pass over the evaluation samples, timing the same transcription that is
scored. The original ran the model twice per sample — once for WER, once for
latency — which doubles the runtime and reports the latency of a call whose
output was thrown away.

In [ ]:
EVAL_SAMPLES = 100

ami_test = load_dataset("edinburghcstr/ami", "ihm", split="test")

wer_scores, latencies = [], []
skipped = 0

for index, sample in enumerate(ami_test.select(range(EVAL_SAMPLES))):
    reference_raw = sample["text"]

    if is_degenerate_sample(reference_raw):
        skipped += 1
        continue

    predicted_raw, seconds = whisper_asr(
        sample["audio"]["array"], sample["audio"]["sampling_rate"]
    )

    reference = normalize_text(reference_raw)
    predicted = normalize_text(predicted_raw)

    sample_wer = wer(reference, predicted)
    wer_scores.append(sample_wer)
    latencies.append(seconds)

    print(f"[{index:3d}] WER {sample_wer:5.2f} | REF {reference[:44]!r}")

avg_wer = sum(wer_scores) / len(wer_scores) if wer_scores else None
avg_latency = sum(latencies) / len(latencies) if latencies else None

print("\n" + "=" * 60)
print(f"Samples scored:      {len(wer_scores)}")
print(f"Samples skipped:     {skipped}  (fewer than 3 reference words)")
if avg_wer is not None:
    # Clamped: a single pathological sample can push WER above 1.0, and a
    # negative "accuracy" is not a result, it is a reporting bug.
    print(f"Average WER:         {avg_wer:.3f}  ({avg_wer * 100:.1f}%)")
    print(f"Accuracy:            {max(0.0, (1 - avg_wer)) * 100:.2f}%")
    print(f"Average latency:     {avg_latency:.3f} s per utterance")
else:
    print("No valid samples to score.")
print("=" * 60)

### 2.2 · Qualitative check on ICSI

ICSI has no aligned reference text in this configuration, so it cannot produce a
WER. It is used here for what it *can* show: how the model behaves on
far-field, overlapping, multi-party audio rather than clean headset audio.

In [ ]:
icsi_sample = next(iter(load_dataset("argmaxinc/icsi-meetings", split="test", streaming=True)))

if "audio" not in icsi_sample or icsi_sample["audio"] is None:
    print("This sample carries no audio.")
else:
    audio = icsi_sample["audio"]["array"]
    rate = icsi_sample["audio"]["sampling_rate"]
    text, seconds = whisper_asr(audio, rate)
    print(f"Transcribed {len(audio) / rate:.1f}s in {seconds:.1f}s\n")
    print(text[:1000])

---

# 3 · Speaker Diarization — pyannote

**Target: DER ≤ 12%.**

Diarization answers *who spoke when*. It is scored with Diarization Error Rate:
missed speech, plus false alarm, plus speaker confusion, over total reference
speech. Overlapping speech is where it is hardest and where meetings live.

**Requires section 1** for `icsi_dataset` and `is_valid_sample`.

`HF_TOKEN` must be set in Colab under 🔑 (left sidebar) → *Add new secret*, and
the model terms accepted with that same account on
[speaker-diarization-3.1](https://hf.co/pyannote/speaker-diarization-3.1) **and**
[segmentation-3.0](https://hf.co/pyannote/segmentation-3.0) — pyannote answers
401 otherwise.

In [ ]:
import numpy as np
import torch
from pyannote.audio import Pipeline
from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import DiarizationErrorRate

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(f"Loading diarization pipeline on: {DEVICE}")
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=HF_TOKEN)

if pipeline is None:
    raise RuntimeError(
        "pyannote returned no pipeline. The token is valid but the model terms "
        "have not been accepted by that account — accept them on both model pages."
    )

pipeline.to(torch.device(DEVICE))
print("Pipeline ready.")

In [ ]:
def build_reference_annotation(sample):
    """Ground-truth speaker turns as a pyannote Annotation."""
    reference = Annotation(uri="meeting_audio")
    for speaker, start, end in zip(
        sample["speakers"], sample["timestamps_start"], sample["timestamps_end"]
    ):
        if end > start:
            reference[Segment(start, end)] = str(speaker)
    return reference


def run_diarization(audio_array, sample_rate):
    """Run the pipeline on an in-memory waveform."""
    tensor = (
        torch.from_numpy(audio_array).float()
        if isinstance(audio_array, np.ndarray)
        else audio_array.float()
    )
    if tensor.ndim == 1:
        tensor = tensor.unsqueeze(0)

    result = pipeline({"waveform": tensor, "sample_rate": sample_rate})

    # pyannote 3.x returns an Annotation; 4.x wraps it in a result object.
    # Calling .itertracks on the 4.x object raises AttributeError, which reads
    # downstream as "diarization produced nothing" on a version bump alone.
    return getattr(result, "speaker_diarization", result)

In [ ]:
MAX_MEETINGS = 5   # each meeting is minutes of audio on a T4; raise deliberately

der_metric = DiarizationErrorRate()
processed = 0

print("Diarization & DER evaluation")
print("=" * 60)

for sample in icsi_dataset:
    if processed >= MAX_MEETINGS:
        break
    if not is_valid_sample(sample):
        continue

    processed += 1
    reference = build_reference_annotation(sample)
    hypothesis = run_diarization(sample["audio"]["array"], sample["audio"]["sampling_rate"])

    sample_der = der_metric(reference, hypothesis)

    print(f"Meeting {processed}:")
    print(f"  Reference turns:  {len(reference)}")
    print(f"  Predicted turns:  {len(hypothesis)}")
    print(f"  DER:              {sample_der * 100:.2f}%")
    for turn, _track, speaker in list(hypothesis.itertracks(yield_label=True))[:3]:
        print(f"    [{turn.start:7.2f}s -> {turn.end:7.2f}s]  {speaker}")
    print("-" * 60)

if processed:
    print(f"\nMeetings evaluated:  {processed}")
    print(f"Cumulative DER:      {abs(der_metric) * 100:.2f}%")
else:
    print("\nNo valid meetings were found — check that section 1 ran.")

---

# 4 · Integration & Privacy

The layer that turns three models into one system: the interface, the upload
flow, the FastAPI service, the merge of ASR + diarization + summarisation, and
the privacy controls governing all of it.

| Requirement | Where it lives |
|---|---|
| UI | `app/web/` — no build step, no framework, no external requests |
| Upload flow | `POST /v1/jobs` — validate, accept, process behind a job id |
| FastAPI backend | `app/main.py` + `app/api/` |
| Integration | `app/pipeline/align_core.py` — every word to its speaker |
| Privacy | `app/core/crypto.py` · `store.py` · `audit.py` · `retention.py` · `pipeline/redaction.py` |

**The architectural point.** Whisper answers *what was said and when*. pyannote
answers *who was speaking and when*. Neither answers *who said what* — and that
is the product. So it is written here, in the integration layer, rather than
inside either model's component. Then every point in the brief is checked
against the transcript before it is shown.

### 4.1 · Fetch the service and run its tests

The tests run against scripted backends holding the same contracts as the real
models: no weights, no network, same answer on any machine. Running them first
is the evidence, before any demo.

In [ ]:
!git clone --depth 1 https://github.com/imzezsv-dot/vocalyze.git vocalyze 2>/dev/null || echo "already cloned"
%cd vocalyze
!pip install -q -r requirements-dev.txt

In [ ]:
!pytest

### 4.2 · Start the service inside the notebook

`TestClient` runs the FastAPI application in this process — the same code that
runs on the server, without binding a port. The encryption key is generated now
and lives only in the environment.

In [ ]:
import json
import os
import sys
import time

sys.path.insert(0, os.getcwd())   # %cd does not add the new directory to the import path

from app.core.crypto import generate_service_key

os.environ["ENCRYPTION_KEY"] = generate_service_key()   # service key, never in the repo
os.environ["DATA_DIR"] = "/content/vocalyze-data"
os.environ["ASR_BACKEND"] = "mock"            # swap to "whisper" for the real model
os.environ["DIARIZATION_BACKEND"] = "mock"    # or "pyannote"
os.environ["SUMMARIZER_BACKEND"] = "mock"     # or "llm"

from fastapi.testclient import TestClient

from app.main import app

client = TestClient(app)
client.__enter__()      # runs the lifespan: the worker and the retention sweeper

print(json.dumps(client.get("/v1/capabilities").json(), indent=2))

`demo_mode: true` means this build is running scripted models. The interface
reads that field and describes itself with it — presenting a scripted sample as
the user's own meeting would be a lie, not a feature.

### 4.3 · The upload flow

Upload is a two-step exchange, not one long request: the file is validated and
accepted, and the work happens behind a job id. A ninety-minute meeting takes
minutes to transcribe, and a POST held open for all of it dies at the first
proxy timeout or dropped connection.

In [ ]:
# A real audio file (a tone) — enough to exercise decoding and 16 kHz normalisation.
!ffmpeg -y -f lavfi -i "sine=frequency=220:duration=12" -ar 44100 -ac 2 /content/meeting.wav -loglevel error

# 1) Upload without recorded consent: refused before the bytes are touched.
refused = client.post(
    "/v1/jobs",
    files={"file": ("meeting.wav", open("/content/meeting.wav", "rb"), "audio/wav")},
    data={"consent": "false"},
)
print(refused.status_code, refused.json()["error"], "->", refused.json()["fix"])

In [ ]:
# 2) The accepted upload.
accepted = client.post(
    "/v1/jobs",
    files={"file": ("meeting.wav", open("/content/meeting.wav", "rb"), "audio/wav")},
    data={"consent": "true"},
).json()

job_id = accepted["job_id"]
token = accepted["access_token"]          # the id alone grants nothing
headers = {"Authorization": f"Bearer {token}"}

print("job:", job_id, "| state:", accepted["state"], "| expires:", accepted["expires_at"])

# 3) Follow the five stages to completion.
while True:
    job = client.get(f"/v1/jobs/{job_id}", headers=headers).json()
    if job["state"] in ("completed", "failed"):
        break
    time.sleep(0.3)

for stage in job["stages"]:
    print(f"  {stage['name']:<13} {stage['state']:<8} {stage.get('detail') or ''}")
print("\nstate:", job["state"], "| error:", job["error"])

### 4.4 · The merge — who said what

The two models' clocks do not agree. Whisper's timings drift at segment edges,
pyannote's turn boundaries land mid-word, and both are wrong during crosstalk.
So attribution is **overlap-maximising**, not boundary-matching: each word goes
to the speaker whose turns cover most of it, with an explicit `UNKNOWN` outcome
instead of a guess when nothing overlaps at all.

In [ ]:
result = client.get(f"/v1/jobs/{job_id}/result", headers=headers).json()
transcript = result["transcript"]

print(f"Speakers: {transcript['speakers']}")
print(f"Duration: {transcript['duration']}s | Lines: {len(transcript['utterances'])}\n")

for utterance in transcript["utterances"][:8]:
    flags = []
    if utterance["overlapped"]:
        flags.append("crosstalk")
    if utterance.get("redacted"):
        flags.append("identifier removed")
    mark = f"  [{', '.join(flags)}]" if flags else ""
    print(
        f"[{utterance['id']:>3}] {utterance['speaker']:<12} "
        f"{utterance['start']:>6.1f}s  {utterance['text'][:64]}{mark}"
    )

### 4.5 · Hallucination control — every point carries its line

A language model asked to summarise a meeting will occasionally produce a
decision nobody made. Prompting reduces that; it does not remove it. So every
generated point is treated as a claim to be checked against the transcript:

1. it must cite utterance ids, and those ids must exist;
2. the claim's content words must appear in the cited lines;
3. every number in the claim must appear in the evidence — invented figures are
   the most damaging failure and the most detectable.

What does not pass is dropped and counted, and the count is shown in the
interface.

In [ ]:
brief = result["brief"]
quality = result["quality"]
by_id = {u["id"]: u["text"] for u in transcript["utterances"]}

print("Summary:", brief["summary"], "\n")

for label, key in [("Decisions", "decisions"), ("Action items", "action_items"), ("Key points", "key_points")]:
    if not brief[key]:
        continue
    print(f"— {label} —")
    for item in brief[key]:
        evidence = item["evidence"]
        owner = f"  (owner: {item['owner']})" if item.get("owner") else ""
        print(f"  • {item['text']}{owner}")
        print(f"    evidence {evidence['utterance_ids']} — match {evidence['grounding']:.0%}")
        for uid in evidence["utterance_ids"]:
            print(f"      {uid}: {by_id.get(uid, '')[:64]}")
    print()

print(f"Points verified: {quality['grounded_claims']} | Points dropped as unsupported: {quality['dropped_claims']}")

The verifier is a standalone unit — no FastAPI, no models — so it can be tested
on its own. Below: a true claim, and one identical to it except for the number.
The second is exactly what has to fail.

In [ ]:
from app.pipeline.grounding_core import Retriever, ground_claim

lines = [
    {"id": "u1", "text": "Word error rate is eleven point two percent on the clean split."},
    {"id": "u2", "text": "We agreed to flag overlapping speech instead of forcing a single speaker."},
]
by_uid = {u["id"]: u for u in lines}
retriever = Retriever(lines)

for claim, cited in [
    ("Word error rate is eleven point two percent on the clean split", ["u1"]),   # true
    ("Word error rate is 47 percent on the clean split",               ["u1"]),   # invented figure
    ("The team decided to cancel the project",                         ["u2"]),   # never said
    ("Overlapping speech is flagged instead of forcing one speaker",   None),     # true, uncited
]:
    verdict = ground_claim(claim, cited, by_uid, retriever)
    print(f"{'ACCEPTED' if verdict.verified else 'REJECTED':<9} ({verdict.score:.2f})  {claim[:56]}")
    if verdict.reason:
        print(f"          reason: {verdict.reason}")

### 4.6 · Privacy requirements, as assertions rather than promises

Privacy is enforced **between** the stages, not at the edges, because that is
where the data actually moves. Raw audio is encrypted the moment it arrives with
a key belonging to that job alone, and destroyed as soon as it has been read.
Identifiers are removed before storage and before any text reaches the
summariser.

In [ ]:
policy = client.get("/v1/privacy/policy").json()
print(json.dumps(policy, indent=2)[:900], "…\n")

privacy = result["privacy"]
print("Audio still stored?     ", privacy["audio_retained"])
print("Encrypted at rest?      ", privacy["encrypted_at_rest"])
print("Consent recorded?       ", privacy["consent"])
print("Retention window (h):   ", privacy["retention_hours"])
print("Models used:            ", privacy["models"])

In [ ]:
# The bytes on disk really are unreadable.
from pathlib import Path

blob = Path(os.environ["DATA_DIR"]) / "jobs" / job_id / "result.enc"
raw = blob.read_bytes()

print("Encrypted file size:", len(raw), "bytes")
print("First 48 bytes:     ", raw[:48])

first_line = transcript["utterances"][0]["text"]
print("\nDoes the meeting text appear inside the file?", first_line.encode() in raw)

In [ ]:
# The audit trail: what the system did with this recording — actions only, never content.
trail = client.get(f"/v1/privacy/jobs/{job_id}/audit", headers=headers).json()
for entry in trail["entries"]:
    print(f"{entry['time'][11:19]}  {entry['action']:<24} {entry['details']}")

# Each line carries the hash of the line before it, so deleting or editing one breaks the chain.
print("\nChain integrity:", client.get("/v1/privacy/audit/verify").json())

In [ ]:
# Redaction is deliberately narrow: budgets, dates and version numbers are the
# substance of a meeting, and over-redaction destroys the transcript.
from app.pipeline.redaction import redact_text

for line in [
    "Email me at layla.ahmed@example.com or call +966 50 123 4567",
    "Error rate is 11.2 percent, the budget is 250000 riyals, and we shipped 3.11.9",
]:
    cleaned, report = redact_text(line)
    print(f"before: {line}")
    print(f"after : {cleaned}")
    print(f"        removed {report.count} {report.by_kind}\n")

In [ ]:
# The right to erasure: blobs are shredded and the job's key destroyed, so any
# surviving copy stays ciphertext forever.
print(client.delete(f"/v1/jobs/{job_id}", headers=headers).json()["message"])

after = client.get(f"/v1/jobs/{job_id}", headers=headers)
print("\nRead after deletion:", after.status_code, after.json()["error"])
print("Any blob left on disk?", list((Path(os.environ["DATA_DIR"]) / "jobs" / job_id).glob("*.enc")))

### 4.7 · How the team's real models plug in

The integration layer never imports Whisper or pyannote directly. It depends on
three contracts in `app/pipeline/` — `ASRBackend`, `DiarizationBackend`,
`SummarizerBackend` — so each component's owner can replace their
implementation without a line changing in the API, the aligner or the interface.
Switching is configuration, not code:

```ini
ASR_BACKEND=whisper                       # section 2's component
DIARIZATION_BACKEND=pyannote              # section 3's component
HUGGINGFACE_TOKEN=hf_…                    # after accepting the pyannote terms
SUMMARIZER_BACKEND=llm                    # the summarisation component
LLM_BASE_URL=http://127.0.0.1:11434/v1    # a local model
```

In [ ]:
import inspect

from app.pipeline.asr import ASRBackend
from app.pipeline.diarization import DiarizationBackend
from app.pipeline.summarizer import SummarizerBackend

for contract in (ASRBackend, DiarizationBackend, SummarizerBackend):
    print(inspect.getsource(contract))

The only requirement is that an implementation returns the shapes defined in
`app/schemas.py`. The one that matters most is `words` inside `ASRResult`: word
timings are what make speaker attribution accurate. A backend that cannot
produce them weakens the whole product — the system then falls back to splitting
a segment proportionally between speakers, and says so through a lower
`speaker_confidence`.

In [ ]:
# A drop-in ASR implementation. Nothing else in the system changes.
from pathlib import Path

from app.schemas import ASRResult, ASRSegment, Word


class MyASR:
    name = "custom"

    def __init__(self, settings=None):
        self.settings = settings

    def transcribe(self, audio_path: Path) -> ASRResult:
        return ASRResult(
            language="en",
            duration=2.0,
            segments=[
                ASRSegment(
                    start=0.0, end=2.0, text="Right, let's start.", confidence=0.93,
                    words=[
                        Word(start=0.0, end=0.9, text="Right,", confidence=0.95),
                        Word(start=1.0, end=2.0, text="let's start.", confidence=0.91),
                    ],
                )
            ],
            model="demo", backend="custom",
        )


# It goes through the production aligner untouched.
from app.pipeline.alignment import build_transcript
from app.schemas import DiarizationResult, SpeakerTurn

turns = DiarizationResult(
    turns=[
        SpeakerTurn(start=0.0, end=0.95, speaker="SPEAKER_00"),
        SpeakerTurn(start=0.95, end=2.0, speaker="SPEAKER_01"),
    ],
    num_speakers=2, backend="custom", model="demo",
)

merged, _stats = build_transcript(MyASR().transcribe(Path("/dev/null")), turns)
for utterance in merged.utterances:
    print(f"{utterance.speaker}: {utterance.text}   (attribution confidence {utterance.speaker_confidence})")

In [ ]:
client.__exit__(None, None, None)   # stop the service and release its resources
print("Service closed.")

---

# 5 · Run it on your own recording

Two ways to demo the finished system on a free Colab GPU. Both need section 4.1
(the clone) to have run.

### 5.1 · Transcribe a file you upload

Real Whisper on your own audio. Speaker separation needs a Hugging Face token —
leave it empty and you still get a full transcription with approximate speaker
separation, which is enough to present.

In [ ]:
!pip install -q faster-whisper pyannote.audio

import importlib
import os
import sys
import time

sys.path.insert(0, os.getcwd())

os.environ["ASR_BACKEND"] = "whisper"          # the real Whisper
os.environ["WHISPER_MODEL"] = "small"          # large-v3 is more accurate and slower
os.environ["DIARIZATION_BACKEND"] = "pyannote"
os.environ["SUMMARIZER_BACKEND"] = "extractive"
os.environ["HUGGINGFACE_TOKEN"] = ""           # <- leave empty, or paste your token

if not os.environ["HUGGINGFACE_TOKEN"]:
    os.environ["DIARIZATION_BACKEND"] = "mock"
    print("No token: your transcription is real, speaker separation is approximate.\n")

# Rebuild the service with the new configuration.
import app.config as config

config.get_settings.cache_clear()
from app.pipeline import registry

registry._cache.clear()
import app.main as main

importlib.reload(main)

from fastapi.testclient import TestClient
from google.colab import files

print("Upload your audio or video file:")
uploaded = files.upload()
name = list(uploaded)[0]

with TestClient(main.app) as service:
    capabilities = service.get("/v1/capabilities").json()
    print(
        f"\nRunning with: asr={capabilities['asr_backend']} "
        f"diarization={capabilities['diarization_backend']}"
    )

    accepted = service.post(
        "/v1/jobs",
        files={"file": (name, uploaded[name], "application/octet-stream")},
        data={"consent": "true"},
    ).json()
    job_id, token = accepted["job_id"], accepted["access_token"]
    headers = {"Authorization": f"Bearer {token}"}

    print("Working — the first run downloads the model weights, so it is slower...")
    while True:
        status = service.get(f"/v1/jobs/{job_id}", headers=headers).json()
        if status["state"] in ("completed", "failed"):
            break
        time.sleep(2)

    if status["state"] == "failed":
        print("\nFailed:", status["error"])
    else:
        result = service.get(f"/v1/jobs/{job_id}/result", headers=headers).json()

        print("\n" + "=" * 72)
        print("Speakers:", result["transcript"]["speakers"])
        print("=" * 72)
        for utterance in result["transcript"]["utterances"]:
            print(f"[{utterance['start']:7.1f}s] {utterance['speaker']:<12} {utterance['text']}")

        print("\n" + "=" * 72)
        print("Summary:", result["brief"]["summary"])
        print("=" * 72)
        for key, label in [("decisions", "decision"), ("action_items", "action"), ("key_points", "point")]:
            for item in result["brief"][key]:
                print(f"• [{label}] {item['text']}")
                print(f"          evidence: {item['evidence']['utterance_ids']}")

### 5.2 · A public link to the live interface

The cell below runs the whole service inside Colab and opens a **public URL**
anyone can open from a phone or a laptop: they upload a recording and get back
an attributed transcript and a verified brief, with real Whisper on the free
GPU.

The link lives as long as the notebook is running. That is enough for a live
demo; a permanent URL needs paid hosting.

In [ ]:
# The whole service, plus a public link. No account needed.
!pip install -q faster-whisper
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import os
import re
import subprocess
import sys
import time

sys.path.insert(0, os.getcwd())

os.environ.update(
    ASR_BACKEND="whisper",             # the real Whisper
    WHISPER_MODEL="small",
    DIARIZATION_BACKEND="mock",        # pyannote needs a token — see 5.1
    SUMMARIZER_BACKEND="extractive",
    DATA_DIR="/content/vocalyze-data",
    MAX_UPLOAD_MB="200",
)

from app.core.crypto import generate_service_key

os.environ["ENCRYPTION_KEY"] = generate_service_key()

service = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("Starting the service...")
time.sleep(10)

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

url = None
for line in tunnel.stdout:
    found = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if found:
        url = found.group(0)
        break

if url:
    print("\n" + "=" * 60)
    print("Live site:", url)
    print("=" * 60)
    print("Open it, upload a recording, get the transcript.")
    print("It works for as long as this notebook is running.")
else:
    print("Could not open the tunnel. Re-run this cell.")